In [1]:
# Modules
import duckdb as db
import pandas as pd
import numpy as np
import jenkspy


In [2]:
# Load data

mr = db.query("SELECT * FROM 'model_result.parquet'").df()


In [3]:
mr

,bovine_id,farm_id,vig_date,contact_order,last_contact,exposure_duration,herd_size,route_size,p_mean_stochastic,p_std,p_min,p_max,p_ci_lower,p_ci_upper,cv,p_deterministic,risk_class
0,B249,F1736,2021-07-29 11:05:54.0000000,4,2020-08-23 00:00:00.000,5,254.200000,4,0.000909,0.000339,0.000282,0.002064,0.000379,0.001632,0.373392,0.000914,Low
1,B641,F762,2023-11-29 07:41:24.0000000,1,2021-01-20 00:00:00.000,1139,530.299809,3,0.206206,0.068186,0.070155,0.413188,0.093166,0.343895,0.330670,0.210117,High
2,B786,F855,2021-09-02 13:18:03.0000000,1,2020-06-21 00:00:00.000,435,200.301149,1,0.072564,0.026060,0.023195,0.158008,0.031061,0.127133,0.359131,0.073323,High
3,B788,F449,2024-01-16 08:47:29.0000000,1,2020-06-04 00:00:00.000,5862,190.325827,1,0.610926,0.135675,0.268898,0.899269,0.343681,0.837110,0.222081,0.638090,High
4,B346,F1757,2021-04-20 16:27:28.0000000,1,2021-02-08 00:00:00.000,37,2428.621622,5,0.009423,0.003503,0.002932,0.021286,0.003940,0.016867,0.371731,0.009473,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1447,B468,F1806,2023-10-19 09:31:24.0000000,1,2023-06-30 00:00:00.000,110,895.245455,5,0.024220,0.008933,0.007582,0.054250,0.010181,0.043140,0.368822,0.024376,Medium
1448,B400,F1812,2025-05-16 17:37:10.0000000,4,2023-07-04 00:00:00.000,193,1441.538860,7,0.044924,0.016384,0.014188,0.099425,0.019029,0.079459,0.364708,0.045272,Medium
1449,B722,F107,2023-08-15 16:48:30.0000000,1,2023-08-14 00:00:00.000,0,1.000000,4,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,NaN,0.000000,Low
1450,B409,F1822,2024-07-30 10:46:47.0000000,1,2023-10-27 00:00:00.000,276,7602.076087,5,0.077402,0.027721,0.024794,0.168058,0.033193,0.135380,0.358145,0.078168,High


In [6]:

# The 'Anchor' (Max Risk per Positive Animal)

# Anchor cases: just only one farm in the route

mrc = mr.query("route_size ==1")

max_risks = mrc.groupby('bovine_id')['p_deterministic'].max()

#  Sensitivity Threshold (T_High)

T_high = max_risks.quantile(0.05)

# Medium/Low 

remaining_contacts = mr[mr['p_deterministic'] < T_high]['p_deterministic']
T_medium = remaining_contacts.median()

print(f"--- Risk Thresholds ---")
print(f"HIGH RISK (> {T_high:.4f}):")
print(f"   Defined as the risk level exceeded by at least one contact in 95% of positive cases.")
print(f"MEDIUM RISK ({T_medium:.4f} - {T_high:.4f}):")
print(f"   defined as the upper 50% of the remaining non-critical contacts.")
print(f"LOW RISK (< {T_medium:.4f}):")
print(f"   Defined as the bottom 50% of background contacts.")

# Validate
def classify_risk(p):
    if p >= T_high: return "High"
    elif p >= T_medium: return "Medium"
    else: return "Low"

mr['risk_class'] = mr['p_deterministic'].apply(classify_risk)

# Validation: How many high-risk farms per animal?


animals_with_high_risk = mr[mr['risk_class'] == 'High']['bovine_id'].nunique()


total_positive_animals = mr['bovine_id'].nunique()


farms_per_animal = mr[mr['risk_class'] == 'High'].groupby('bovine_id').size()

print(f"\n Validation")
print(f"Model Sensitivity (Animals detected): {animals_with_high_risk / total_positive_animals:.1%}")
print(f"Operational Load: {farms_per_animal.mean():.2f} High-Risk farms flagged per animal.")

--- Risk Thresholds ---
HIGH RISK (> 0.0518):
   Defined as the risk level exceeded by at least one contact in 95% of positive cases.
MEDIUM RISK (0.0034 - 0.0518):
   defined as the upper 50% of the remaining non-critical contacts.
LOW RISK (< 0.0034):
   Defined as the bottom 50% of background contacts.

 Validation
Model Sensitivity (Animals detected): 96.8%
Operational Load: 1.56 High-Risk farms flagged per animal.


In [7]:
# Saving mr object

db.query("SELECT * FROM mr").to_parquet("model_result.parquet")